##### Baixar os dados do Bosla Família os meses de jan/26 a mar/26 Pela API

##### Filtrar os dados pelo município de São Paulo, analisando quanto foi pago em cada mês e o total pago no período.

##### Contar quantos benefícios foram recebidos por cada UF e qual a UF que mais recebeu benefícios no período.

#### Imports

In [ ]:
%pip install requests
%pip install pandas

In [1]:
import requests
import pandas as pd
from zipfile import ZipFile
from io import BytesIO

#### Função para Get

In [14]:
def baixar_e_extrair_dados(url):

    # Envia uma requisição para baixar o arquivo
    response = requests.get(url, timeout=30)

    # Verifica se a requisição foi bem-sucedida
    if response.status_code == 200:

        # Abre o arquivo zip em memória
        with ZipFile(BytesIO(response.content)) as arquivo_zip:
            nomes_arquivos = arquivo_zip.namelist()

            if not nomes_arquivos:
                print("Arquivo ZIP vazio.")
                return None

            # Assume que há apenas um arquivo e extrai
            with arquivo_zip.open(nomes_arquivos[0]) as arquivo_csv:
                df = pd.read_csv(arquivo_csv, sep=';', encoding='ISO-8859-1')
                return df

    else:
        print(f"Falha ao baixar o arquivo. Status code: {response.status_code}")
        return None


#### Filtra os meses

In [15]:
meses_2026 = [f"2026{mes:02d}" for mes in range(1, 13)]
meses_2026

['202601',
 '202602',
 '202603',
 '202604',
 '202605',
 '202606',
 '202607',
 '202608',
 '202609',
 '202610',
 '202611',
 '202612']

#### Faz chamada na API recursiva e salva os meses

In [16]:
dfs_2026 = []
for mes in meses_2026:
    url = f"https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/{mes}"
    df_mes = baixar_e_extrair_dados(url)
    if df_mes is not None:
        df_mes["MES"] = mes
        dfs_2026.append(df_mes)

df = pd.concat(dfs_2026, ignore_index=True)
df.head()

Falha ao baixar o arquivo. Status code: 403
Falha ao baixar o arquivo. Status code: 403
Falha ao baixar o arquivo. Status code: 403
Falha ao baixar o arquivo. Status code: 403


,MÊS COMPETÊNCIA,MÊS REFERÊNCIA,UF,CÓDIGO MUNICÍPIO SIAFI,NOME MUNICÍPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA,MES
0,202601,202502,MG,5041,PONTE NOVA,NaN,1.652367e+10,MILENE APARECIDA SILVA DE ARAUJO,"850,00",202601
1,202601,202502,RO,5,JI-PARANA,***.582.082-**,1.622464e+10,GLEISIMAR LIMA SILVA,"750,00",202601
2,202601,202503,MG,5041,PONTE NOVA,NaN,1.652367e+10,MILENE APARECIDA SILVA DE ARAUJO,"850,00",202601
3,202601,202505,SP,2965,BERTIOGA,***.740.014-**,2.061952e+10,LIGIA SAMARA LIMA DOS SANTOS,"800,00",202601
4,202601,202506,AC,643,ACRELANDIA,***.124.312-**,1.639916e+10,CLEICIANE SOUZA RIBEIRO,"850,00",202601


#### Printa os meses para conferência do que retornou

In [19]:
# Meses distintos nas colunas de período
for col in ["MÊS COMPETÊNCIA", "MÊS REFERÊNCIA", "MES"]:
    meses = sorted(df[col].dropna().unique())
    print(f"{col}: {len(meses)} meses | min={meses[0]} | max={meses[-1]}")
    print(meses, "\n")

MÊS COMPETÊNCIA: 8 meses | min=202601 | max=202608
[np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608)] 

MÊS REFERÊNCIA: 18 meses | min=202502 | max=202608
[np.int64(202502), np.int64(202503), np.int64(202505), np.int64(202506), np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511), np.int64(202512), np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608)] 

MES: 8 meses | min=202601 | max=202608
['202601', '202602', '202603', '202604', '202605', '202606', '202607', '202608'] 



#### Colunas necessárias + Conversão do VALOR PARCELA

In [26]:
df_colunas_necessarias = df[[
    "MES",
    "MÊS COMPETÊNCIA",
    "UF",
    "NOME MUNICÍPIO",
    "VALOR PARCELA"
]].copy()

# "850,00" -> 850.0 (necessário para somar e tirar médias)
df_colunas_necessarias["VALOR PARCELA"] = (
    df_colunas_necessarias["VALOR PARCELA"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_colunas_necessarias.info()
df_colunas_necessarias.head()

<class 'pandas.DataFrame'>
RangeIndex: 152692800 entries, 0 to 152692799
Data columns (total 5 columns):
 #   Column           Dtype  
---  ------           -----  
 0   MES              str    
 1   MÊS COMPETÊNCIA  int64  
 2   UF               str    
 3   NOME MUNICÍPIO   str    
 4   VALOR PARCELA    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 5.7 GB


,MES,MÊS COMPETÊNCIA,UF,NOME MUNICÍPIO,VALOR PARCELA
0,202601,202601,MG,PONTE NOVA,850.0
1,202601,202601,RO,JI-PARANA,750.0
2,202601,202601,MG,PONTE NOVA,850.0
3,202601,202601,SP,BERTIOGA,800.0
4,202601,202601,AC,ACRELANDIA,850.0


In [27]:
df_colunas_necessarias

,MES,MÊS COMPETÊNCIA,UF,NOME MUNICÍPIO,VALOR PARCELA
0,202601,202601,MG,PONTE NOVA,850.0
1,202601,202601,RO,JI-PARANA,750.0
2,202601,202601,MG,PONTE NOVA,850.0
3,202601,202601,SP,BERTIOGA,800.0
4,202601,202601,AC,ACRELANDIA,850.0
...,...,...,...,...,...
152692795,202608,202608,TO,XAMBIOA,750.0
152692796,202608,202608,TO,XAMBIOA,650.0
152692797,202608,202608,TO,XAMBIOA,600.0
152692798,202608,202608,TO,XAMBIOA,600.0
